# ⚡ polars-anta: 1-Minute Interactive Quickstart & Benchmark
### Ultra-fast, zero-C-dependency technical analysis and alpha factor library for Polars.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huuan26/polars-anta/blob/master/examples/quickstart_colab.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-huuan26%2Fpolars--anta-blue?logo=github)](https://github.com/huuan26/polars-anta)
[![PyPI](https://img.shields.io/pypi/v/polars-anta.svg?color=blue)](https://pypi.org/project/polars-anta/)
[![License](https://img.shields.io/badge/license-Apache%202.0-blue.svg)](https://github.com/huuan26/polars-anta/blob/master/LICENSE)

> ⭐ **If you find `polars-anta` fast and useful, please consider starring the repository on [GitHub](https://github.com/huuan26/polars-anta)!**

---
### Why quantitative researchers choose `polars-anta`:
1. **Zero C/GCC dependencies**: Unlike TA-Lib, `pip install polars-anta` runs on any OS immediately without complex compilers.
2. **Powered by Rust Expressions**: All calculations push directly into Polars' parallel Rust engine.
3. **Multi-Asset Scaling**: Compute across thousands of symbols simultaneously using `.over("ticker")` with zero Python for-loops.
4. **Live 112x Speedup**: Witness the benchmark comparison vs Pandas in Section 3 below!

In [ ]:
# Step 1: Install polars-anta and dependencies in Colab (< 5 seconds)
!pip install -q "polars>=1.0.0" polars-anta matplotlib pandas tabulate

In [ ]:
# Step 2: Import and verify installation
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import polars_anta

print(f"✅ Polars Version       : {pl.__version__}")
print(f"✅ polars-anta Version   : {polars_anta.__version__}")
print(f"✅ Registered Indicators : {len(polars_anta.__all__)} technical indicators & alpha transforms")

---
## 1. Single-Asset Technical Analysis (Chained Expressions)

`polars-anta` registers both `.ta` and `.anta` expression namespaces. You can chain indicators seamlessly within standard `.with_columns()` or lazy query plans:

In [ ]:
# Generate realistic sample market data (1,000 bars)
np.random.seed(42)
n_bars = 1000
returns = np.random.normal(0.0003, 0.015, n_bars)
close = 150.0 * np.exp(np.cumsum(returns))
high = close * (1.0 + np.abs(np.random.normal(0, 0.005, n_bars)))
low = close * (1.0 - np.abs(np.random.normal(0, 0.005, n_bars)))
open_ = (high + low) / 2.0
volume = np.random.randint(1000, 50000, n_bars).astype(float)

df = pl.DataFrame({
    "open": open_,
    "high": high,
    "low": low,
    "close": close,
    "volume": volume,
})

# Compute multiple indicators simultaneously in a single pass:
# Multi-output indicators (Bollinger Bands, MACD) return clean typed Structs
result_df = df.with_columns([
    pl.col("close").ta.sma(20).alias("sma_20"),
    pl.col("close").ta.ema(50).alias("ema_50"),
    pl.col("close").ta.rsi(14).alias("rsi_14"),
    pl.col("close").ta.zscore(20).alias("zscore_20"),
    pl.col("close").ta.bollinger_bands(20, 2.0).alias("bb"),
    pl.col("close").ta.macd().alias("macd"),
]).unnest(["bb", "macd"])

print("📊 Computed Technical Analysis Table (Latest 5 Bars):")
result_df.select([
    "close", "sma_20", "rsi_14", "zscore_20", "upper", "lower", "macd", "histogram"
]).tail(5)

---
## 2. Multi-Asset Portfolio Parallelization (`.over("ticker")`)

The biggest pain point in quantitative research is computing indicators across 500+ tickers.
- In **Pandas**, researchers must write slow for-loops or `.groupby().apply()`.
- In **polars-anta**, simply append `.over("ticker")` and Polars automatically parallelizes across all available CPU threads in Rust!

In [ ]:
# Simulate an investment universe: 500 tickers x 1,000 bars = 500,000 rows
n_tickers = 500
n_bars_universe = 1000
total_rows = n_tickers * n_bars_universe

tickers = [f"SYM_{i:03d}" for i in range(n_tickers)]
ticker_col = np.repeat(tickers, n_bars_universe)
close_prices = np.random.uniform(20.0, 250.0, total_rows)
volumes = np.random.uniform(10_000, 1_000_000, total_rows)

universe_df = pl.DataFrame({
    "ticker": ticker_col,
    "close": close_prices,
    "volume": volumes,
})

print(f"Universe size: {n_tickers} tickers x {n_bars_universe:,} bars = {total_rows:,} rows")

t0 = time.perf_counter()

# Parallel execution across all CPU cores in Rust:
universe_features = universe_df.with_columns([
    pl.col("close").ta.sma(20).over("ticker").alias("sma_20"),
    pl.col("close").ta.rsi(14).over("ticker").alias("rsi_14"),
    pl.col("close").ta.zscore(20).over("ticker").alias("zscore_20"),
])

elapsed = time.perf_counter() - t0
throughput = total_rows / elapsed

print(f"\n⚡ Computed SMA, RSI, & Z-Score across 500 tickers in {elapsed * 1000:.2f} ms ({elapsed:.4f} s)!")
print(f"🚀 Throughput: {throughput:,.0f} rows/second on standard Colab CPU")

---
## 3. 🔥 Live Benchmark: polars-anta vs Pandas (100x+ Speedup)

Let's test **Rolling Linear Regression Slope (period=14)** on 200,000 bars right here inside your Google Colab instance.
- **Pandas**: Uses `.rolling(14).apply(...)` (single-threaded Python GIL bottleneck).
- **polars-anta**: Executes analytical expression math compiled in Rust.

In [ ]:
n_bench = 200_000
np.random.seed(42)
bench_prices = 100.0 * np.exp(np.cumsum(np.random.normal(0.0001, 0.01, n_bench)))

df_bench_pd = pd.DataFrame({"close": bench_prices})
df_bench_pl = pl.DataFrame({"close": bench_prices})

print(f"Running head-to-head benchmark on {n_bench:,} bars...\n")

# 1. Pandas Benchmark
t0 = time.perf_counter()
w = list(range(1, 15))
w_sum = sum(w)
wma_pd = df_bench_pd["close"].rolling(14).apply(lambda x: np.dot(x, w) / w_sum, raw=True)
sma_pd = df_bench_pd["close"].rolling(14).mean()
slope_pd = (wma_pd - sma_pd) * (6.0 / 13.0)
t_pd = time.perf_counter() - t0

# 2. polars-anta Benchmark
t0 = time.perf_counter()
slope_pl = df_bench_pl.select(pl.col("close").ta.linear_regression_slope(14))
t_pl = time.perf_counter() - t0

speedup = t_pd / t_pl

print("=" * 60)
print(f"🏆 LIVE BENCHMARK RESULTS ({n_bench:,} bars):")
print(f"   • Pandas (rolling + apply) : {t_pd:.4f} seconds")
print(f"   • polars-anta (Rust Expr)  : {t_pl:.4f} seconds")
print(f"   🚀 Measured Speedup        : {speedup:.1f}x FASTER!")
print("=" * 60)

---
## 4. Publication-Ready Visualizations

Visualize the computed price action, Bollinger Bands, and MACD:

In [ ]:
plot_data = result_df.tail(150).to_pandas()

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2.5, 1]}
)

# Panel 1: Price and Bollinger Bands
ax1.plot(plot_data.index, plot_data["close"], label="Close Price", color="#1f77b4", lw=1.6)
ax1.plot(plot_data.index, plot_data["sma_20"], label="SMA (20)", color="#ff7f0e", linestyle="--", lw=1.2)
ax1.plot(plot_data.index, plot_data["upper"], label="Upper BB", color="#2ca02c", alpha=0.8, lw=1)
ax1.plot(plot_data.index, plot_data["lower"], label="Lower BB", color="#d62728", alpha=0.8, lw=1)
ax1.fill_between(plot_data.index, plot_data["lower"], plot_data["upper"], color="#2ca02c", alpha=0.08)
ax1.set_title("polars-anta: Native Polars Technical Indicator Visualizer", fontsize=13, fontweight="bold")
ax1.set_ylabel("Price ($)", fontsize=11)
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.25)

# Panel 2: MACD and Histogram
ax2.plot(plot_data.index, plot_data["macd"], label="MACD", color="#1f77b4", lw=1.2)
ax2.plot(plot_data.index, plot_data["signal"], label="Signal", color="#ff7f0e", lw=1.2)
bar_colors = ["#2ca02c" if h >= 0 else "#d62728" for h in plot_data["histogram"]]
ax2.bar(plot_data.index, plot_data["histogram"], label="Hist", color=bar_colors, alpha=0.5, width=0.8)
ax2.set_ylabel("MACD", fontsize=11)
ax2.set_xlabel("Bar Index", fontsize=11)
ax2.legend(loc="upper left")
ax2.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

---
## 5. Next Steps & GitHub Star

If you enjoyed this demo and want to support the project:
- ⭐️ **Star the repo on GitHub**: [huuan26/polars-anta](https://github.com/huuan26/polars-anta)
- 📖 **Read the Documentation**: [huuan26.github.io/polars-anta](https://huuan26.github.io/polars-anta/)
- 📦 **Install locally**: `pip install polars-anta`
- 💬 **Feedback & Issues**: [GitHub Issues](https://github.com/huuan26/polars-anta/issues)